# 04b - Retrieval Weight Sweep and Candidate Recall

Runs a light optimization pass after the first retrieval ablation. It measures @30 candidate recall and tests hybrid dense/BM25 weights without re-embedding each query for every weight.

In [ ]:
from pathlib import Path
import json
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
benchmark_csv = DRIVE_ROOT / config['benchmark_csv']
index_root = DRIVE_ROOT / config.get('official_index_root', 'indexes/official_law_v3')
output_dir = DRIVE_ROOT / 'outputs/retrieval_eval'

for path in [benchmark_csv, index_root / 'index_manifest.json']:
    if not path.exists():
        raise FileNotFoundError(path)

benchmark_csv, index_root, output_dir

In [ ]:
import torch
from src.evaluation_retrieval_optimization import evaluate_weight_sweep

device = 'cuda' if torch.cuda.is_available() else 'cpu'
summary_df = evaluate_weight_sweep(
    benchmark_csv=benchmark_csv,
    index_root=index_root,
    output_dir=output_dir,
    candidate_k=30,
    device=device,
    dense_weights=[0.0, 0.25, 0.5, 0.55, 0.65, 0.75, 0.85, 0.9, 1.0],
)

summary_df

In [ ]:
display_columns = [
    'mode', 'dense_weight', 'bm25_weight', 'question_count',
    'doc_hit@5', 'doc_hit@10', 'doc_hit@30',
    'article_hit@5', 'article_hit@10', 'article_hit@30',
    'article_mrr', 'article_ndcg@5', 'article_ndcg@10', 'article_ndcg@30',
]
summary_df[display_columns]

Use the best `article_hit@5` / `article_mrr` setting for the reranker candidate generator. If `article_hit@30` is meaningfully higher than `article_hit@5`, a reranker has room to improve top-5 article accuracy.